In [1]:
# Imports
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

/home/md-ismail-quraishi/ismail/personality-development/learning/ai-ml/bert/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import torch
import numpy as np

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
from tqdm.auto import tqdm

In [3]:
# Download the IMDb dataset
dataset = load_dataset("stanfordnlp/imdb")
# Display dataset structure
print(dataset)
# Display one training example
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and

In [4]:
dataset["train"][3]

{'text': "This film was probably inspired by Godard's Masculin, féminin and I urge you to see that film instead.<br /><br />The film has two strong elements and those are, (1) the realistic acting (2) the impressive, undeservedly good, photo. Apart from that, what strikes me most is the endless stream of silliness. Lena Nyman has to be most annoying actress in the world. She acts so stupid and with all the nudity in this film,...it's unattractive. Comparing to Godard's film, intellectuality has been replaced with stupidity. Without going too far on this subject, I would say that follows from the difference in ideals between the French and the Swedish society.<br /><br />A movie of its time, and place. 2/10.",
 'label': 0}

In [5]:
df = pd.DataFrame(dataset["train"])

print(df.head())
print("\nDataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nLabel distribution:")
print(df["label"].value_counts())

                                                text  label
0  I rented I AM CURIOUS-YELLOW from my video sto...      0
1  "I Am Curious: Yellow" is a risible and preten...      0
2  If only to avoid making this type of film in t...      0
3  This film was probably inspired by Godard's Ma...      0
4  Oh, brother...after hearing about this ridicul...      0

Dataset shape: (25000, 2)

Missing values:
text     0
label    0
dtype: int64

Label distribution:
label
0    12500
1    12500
Name: count, dtype: int64


In [6]:
MODEL_NAME = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
review = "This movie was absolutely amazing!"
encoded = tokenizer(
    review,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))

['[CLS]', 'this', 'movie', 'was', 'absolutely', 'amazing', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', 

In [7]:
df.tail()

,text,label
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1
24999,The story centers around Barry McKenzie who mu...,1


In [9]:
# -----------------------------
# 1. Configuration
# -----------------------------

MODEL_NAME = "google-bert/bert-base-uncased"

TRAIN_SAMPLES = 2000
TEST_SAMPLES = 500
MAX_LENGTH = 128
BATCH_SIZE = 4
EPOCHS = 1
LEARNING_RATE = 2e-5

DEVICE = torch.device("cpu")

MODEL_DIR = os.path.join("models", "bert-sentiment")
os.makedirs(MODEL_DIR, exist_ok=True)

print("Device:", DEVICE)

# -----------------------------
# 2. Load the IMDb dataset
# -----------------------------

print("\nLoading IMDb dataset...")

dataset = load_dataset("stanfordnlp/imdb")

# Shuffle before selecting samples so that
# the subset is not dominated by one class.
train_data = (
    dataset["train"]
    .shuffle(seed=42)
    .select(range(TRAIN_SAMPLES))
)

test_data = (
    dataset["test"]
    .shuffle(seed=42)
    .select(range(TEST_SAMPLES))
)

print("Training reviews:", len(train_data))
print("Testing reviews:", len(test_data))

# -----------------------------
# 3. Load the tokenizer
# -----------------------------

print("\nLoading BERT tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# -----------------------------
# 4. Tokenize the reviews
# -----------------------------

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )


print("\nTokenizing training data...")

train_data = train_data.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

print("Tokenizing test data...")

test_data = test_data.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

# ERROR SOLVED to rename label
train_data = train_data.rename_column("label", "labels")
test_data = test_data.rename_column("label", "labels")

# Convert dataset columns to PyTorch tensors.
train_data.set_format("torch")
test_data.set_format("torch")

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# -----------------------------
# 5. Load pretrained BERT
# -----------------------------

print("\nLoading pretrained BERT model...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

model.to(DEVICE)

# AdamW updates the model parameters.
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)

# -----------------------------
# 6. Train the model
# -----------------------------

print("\nStarting training...")

for epoch in range(EPOCHS):
    model.train()

    total_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
    )

    for batch in progress_bar:
        # Move each batch to the selected device.
        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        # Clear gradients from the previous step.
        optimizer.zero_grad()

        # BERT calculates the classification loss
        # because the batch contains labels.
        outputs = model(**batch)

        loss = outputs.loss

        # Calculate gradients.
        loss.backward()

        # Update model parameters.
        optimizer.step()

        total_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1} - "
        f"Average training loss: {average_loss:.4f}"
    )

# -----------------------------
# 7. Evaluate the model
# -----------------------------

print("\nEvaluating model...")

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        labels = batch["labels"]

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=1,
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

# Calculate evaluation metrics.
accuracy = accuracy_score(
    all_labels,
    all_predictions,
)

precision = precision_score(
    all_labels,
    all_predictions,
    zero_division=0,
)

recall = recall_score(
    all_labels,
    all_predictions,
    zero_division=0,
)

f1 = f1_score(
    all_labels,
    all_predictions,
    zero_division=0,
)

print("\n========== MODEL EVALUATION ==========")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\nClassification report:")

print(
    classification_report(
        all_labels,
        all_predictions,
        labels=[0, 1],
        target_names=["Negative", "Positive"],
        zero_division=0,
    )
)

# -----------------------------
# 8. Save the trained model
# -----------------------------

print("\nSaving model...")

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print(f"Model saved to: {MODEL_DIR}")
print("\nTraining and evaluation completed!")

Device: cpu

Loading IMDb dataset...
Training reviews: 2000
Testing reviews: 500

Loading BERT tokenizer...

Tokenizing training data...
Tokenizing test data...

Loading pretrained BERT model...


Loading weights: 100%|█████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 3999.21it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING


Starting training...


Epoch 1/1: 100%|██████████████████████████████████████████████████████████████| 500/500 [1:33:56<00:00, 11.27s/it, loss=0.2364]


Epoch 1 - Average training loss: 0.4943

Evaluating model...


Testing: 100%|███████████████████████████████████████████████████████████████████████████████| 125/125 [06:33<00:00,  3.15s/it]



========== MODEL EVALUATION ==========
Accuracy:  0.8280
Precision: 0.7667
Recall:    0.9350
F1-score:  0.8425

Classification report:
              precision    recall  f1-score   support

    Negative       0.92      0.72      0.81       254
    Positive       0.77      0.93      0.84       246

    accuracy                           0.83       500
   macro avg       0.84      0.83      0.83       500
weighted avg       0.84      0.83      0.83       500


Saving model...


Writing model shards: 100%|██████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.85it/s]


Model saved to: models/bert-sentiment

Training and evaluation completed!


In [17]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Path to your saved model
MODEL_PATH = "models/bert-sentiment"

# Load the trained model and tokenizer
print("Loading trained model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

# Use CPU
device = torch.device("cpu")
model.to(device)
model.eval()


def predict_sentiment(review):
    # Convert review into tokens that BERT understands
    inputs = tokenizer(
        review,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128,
    )

    # Move inputs to the CPU
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Predict without calculating gradients
    with torch.no_grad():
        outputs = model(**inputs)

    # Convert model scores into probabilities
    probabilities = torch.softmax(outputs.logits, dim=1)[0]

    # IMDb dataset labels: 0 = Negative, 1 = Positive
    predicted_class = torch.argmax(probabilities).item()
    sentiment = "Positive" if predicted_class == 1 else "Negative"
    confidence = probabilities[predicted_class].item() * 100

    print(f"\nReview: {review}")
    print(f"Predicted sentiment: {sentiment}")
    print(f"Confidence: {confidence:.2f}%")

    return sentiment


if __name__ == "__main__":
    review = input("Enter a movie review: ")
    predict_sentiment(review)

Loading trained model...


Loading weights: 100%|█████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 3224.14it/s]


Enter a movie review:  better than a bad movie



Review: better than a bad movie
Predicted sentiment: Positive
Confidence: 56.42%
